In [1]:
!pip install nltk

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from collections import Counter
from torch.utils.data import Dataset, DataLoader
from nltk.tokenize import word_tokenize
import nltk

In [50]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("ukveteran/big-text")

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/big-text


In [62]:
with open('/kaggle/input/big-text/big.txt', 'r') as f:
  document = f.read()

In [65]:
document

'The Project Gutenberg EBook of The Adventures of Sherlock Holmes\nby Sir Arthur Conan Doyle\n(#15 in our series by Sir Arthur Conan Doyle)\n\nCopyright laws are changing all over the world. Be sure to check the\ncopyright laws for your country before downloading or redistributing\nthis or any other Project Gutenberg eBook.\n\nThis header should be the first thing seen when viewing this Project\nGutenberg file.  Please do not remove it.  Do not change or edit the\nheader without written permission.\n\nPlease read the "legal small print," and other information about the\neBook and Project Gutenberg at the bottom of this file.  Included is\nimportant information about your specific rights and restrictions in\nhow the file may be used.  You can also find out about how to make a\ndonation to Project Gutenberg, and how to get involved.\n\n\n**Welcome To The World of Free Plain Vanilla Electronic Texts**\n\n**eBooks Readable By Both Humans and By Computers, Since 1971**\n\n*****These eBooks 

In [4]:
# Tokenization
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [66]:
# tokenize
tokens = word_tokenize(document.lower())

In [67]:
# build vocab
vocab = {'<oov>':0}

for token in Counter(tokens).keys():
  if token not in vocab:
    vocab[token] = len(vocab)

vocab

{'<oov>': 0,
 'the': 1,
 'project': 2,
 'gutenberg': 3,
 'ebook': 4,
 'of': 5,
 'adventures': 6,
 'sherlock': 7,
 'holmes': 8,
 'by': 9,
 'sir': 10,
 'arthur': 11,
 'conan': 12,
 'doyle': 13,
 '(': 14,
 '#': 15,
 '15': 16,
 'in': 17,
 'our': 18,
 'series': 19,
 ')': 20,
 'copyright': 21,
 'laws': 22,
 'are': 23,
 'changing': 24,
 'all': 25,
 'over': 26,
 'world': 27,
 '.': 28,
 'be': 29,
 'sure': 30,
 'to': 31,
 'check': 32,
 'for': 33,
 'your': 34,
 'country': 35,
 'before': 36,
 'downloading': 37,
 'or': 38,
 'redistributing': 39,
 'this': 40,
 'any': 41,
 'other': 42,
 'header': 43,
 'should': 44,
 'first': 45,
 'thing': 46,
 'seen': 47,
 'when': 48,
 'viewing': 49,
 'file': 50,
 'please': 51,
 'do': 52,
 'not': 53,
 'remove': 54,
 'it': 55,
 'change': 56,
 'edit': 57,
 'without': 58,
 'written': 59,
 'permission': 60,
 'read': 61,
 '``': 62,
 'legal': 63,
 'small': 64,
 'print': 65,
 ',': 66,
 "''": 67,
 'and': 68,
 'information': 69,
 'about': 70,
 'at': 71,
 'bottom': 72,
 'inclu

In [68]:
len(vocab)

36721

In [69]:
input_sentences = document.split('\n')

In [70]:
def text_to_indices(sentence, vocab):

  numerical_sentence = []

  for token in sentence:
    if token in vocab:
      numerical_sentence.append(vocab[token])
    else:
      numerical_sentence.append(vocab['<oov>'])

  return numerical_sentence


In [71]:
input_numerical_sentences = []

for sentence in input_sentences:
  input_numerical_sentences.append(text_to_indices(word_tokenize(sentence.lower()), vocab))


In [72]:
len(input_numerical_sentences)

128458

In [128]:
sentence = input_numerical_sentences[6]
for i in range(1, len(sentence)):
  print(sentence[:i+1])

[40, 38]
[40, 38, 41]
[40, 38, 41, 42]
[40, 38, 41, 42, 2]
[40, 38, 41, 42, 2, 3]
[40, 38, 41, 42, 2, 3, 4]
[40, 38, 41, 42, 2, 3, 4, 28]


In [139]:
training_sequence = []
for sentence in input_numerical_sentences:

  for i in range(1, len(sentence)):
    training_sequence.append(sentence[:i+1])

In [146]:
max = []
for i, seq in enumerate(training_sequence):
  max.append(len(seq))


In [117]:
del training_sequence[35137]

In [155]:
max_len = torch.max(torch.tensor(max, dtype = torch.long))

In [142]:
training_sequence = training_sequence[:30000]

In [143]:
len(training_sequence)

30000

In [136]:
training_sequence[:5]

[1, 2]

In [149]:
training_sequence[0]

[1, 2]

In [156]:
padded_training_sequence = []
for sequence in training_sequence:

  padded_training_sequence.append([0]*(max_len - len(sequence)) + sequence)

In [157]:
len(padded_training_sequence[10])

314

In [158]:
padded_training_sequence = torch.tensor(padded_training_sequence, dtype=torch.long)

In [159]:
padded_training_sequence

tensor([[  0,   0,   0,  ...,   0,   1,   2],
        [  0,   0,   0,  ...,   1,   2,   3],
        [  0,   0,   0,  ...,   2,   3,   4],
        ...,
        [  0,   0,   0,  ...,  82, 190, 974],
        [  0,   0,   0,  ..., 190, 974,   5],
        [  0,   0,   0,  ..., 974,   5,   1]])

In [160]:
X = padded_training_sequence[:, :-1]
y = padded_training_sequence[:,-1]

In [161]:
X

tensor([[  0,   0,   0,  ...,   0,   0,   1],
        [  0,   0,   0,  ...,   0,   1,   2],
        [  0,   0,   0,  ...,   1,   2,   3],
        ...,
        [  0,   0,   0,  ..., 188,  82, 190],
        [  0,   0,   0,  ...,  82, 190, 974],
        [  0,   0,   0,  ..., 190, 974,   5]])

In [162]:
y

tensor([  2,   3,   4,  ..., 974,   5,   1])

In [163]:
class CustomDataset(Dataset):

  def __init__(self, X, y):
    self.X = X
    self.y = y

  def __len__(self):
    return self.X.shape[0]

  def __getitem__(self, idx):
    return self.X[idx], self.y[idx]

In [164]:
dataset = CustomDataset(X,y)

In [165]:
len(dataset)

30000

In [166]:
dataloader = DataLoader(
    dataset,
    batch_size=256,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)


In [167]:
from random import vonmisesvariate
# LSTM Neural Network
class LSTMModel(nn.Module):

  def __init__(self, vocab_size):
    super().__init__()

    self.embedding = nn.Embedding(num_embeddings = vocab_size, embedding_dim = 100)
    self.lstm = nn.LSTM(input_size = 100, hidden_size = 150, batch_first = True)
    self.fc = nn.Linear(150, vocab_size)

  def forward(self, x):
    embedded = self.embedding(x)
    intermediate_hidden_states, (final_hidden_state, final_tuple_state) = self.lstm(embedded)
    output = self.fc(final_hidden_state.squeeze(0))

    return output

In [168]:
# prompt: write code to check gpu/tpu and assign to device

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


In [169]:
model = LSTMModel(len(vocab))
model.to(device)

LSTMModel(
  (embedding): Embedding(36721, 100)
  (lstm): LSTM(100, 150, batch_first=True)
  (fc): Linear(in_features=150, out_features=36721, bias=True)
)

In [170]:
# prompt: write code to see model summary and trainable parameter count

print(model)
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal parameters: {total_params}")
print(f"Trainable parameters: {trainable_params}")


LSTMModel(
  (embedding): Embedding(36721, 100)
  (lstm): LSTM(100, 150, batch_first=True)
  (fc): Linear(in_features=150, out_features=36721, bias=True)
)

Total parameters: 9368171
Trainable parameters: 9368171


In [172]:
epochs = 20
learning_rate = 0.001

In [173]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr = learning_rate)

In [181]:
# training loop

for epoch in range(epochs):
  total_loss = 0

  for batch_x, batch_y in dataloader:

    batch_x, batch_y = batch_x.to(device), batch_y.to(device)

    optimizer.zero_grad()

    output = model(batch_x)

    loss = criterion(output, batch_y)

    loss.backward()

    optimizer.step()

    total_loss = total_loss + loss.item()

  print(f"Epoch: {epoch + 1}, Loss: {total_loss:.4f}")

Epoch: 1, Loss: 271.7805
Epoch: 2, Loss: 265.7451
Epoch: 3, Loss: 258.9334
Epoch: 4, Loss: 253.8055
Epoch: 5, Loss: 247.7766
Epoch: 6, Loss: 241.8503
Epoch: 7, Loss: 236.2666
Epoch: 8, Loss: 230.6380
Epoch: 9, Loss: 225.0236
Epoch: 10, Loss: 220.0681
Epoch: 11, Loss: 214.5151
Epoch: 12, Loss: 210.4503
Epoch: 13, Loss: 204.9621
Epoch: 14, Loss: 199.7260
Epoch: 15, Loss: 195.0433
Epoch: 16, Loss: 191.2127
Epoch: 17, Loss: 186.0774
Epoch: 18, Loss: 181.9654
Epoch: 19, Loss: 177.4078
Epoch: 20, Loss: 173.0793


In [175]:
# prediction

def prediction(model, vocab, text):

  # tokenize
  tokenized_text = word_tokenize(text.lower())

  # text -> numerical indices
  numerical_text = text_to_indices(tokenized_text, vocab)

  # padding
  padded_text = torch.tensor([0] * (61 - len(numerical_text)) + numerical_text, dtype=torch.long).unsqueeze(0)

  padded_text = padded_text.to(device)

  # send to model
  output = model(padded_text)

  # Convert logits to probs
  probs = torch.nn.functional.softmax(output, 1)

  # Find index of max prob
  value, index = torch.max(probs, dim = 1)

  # merge with text
  return text + " " + list(vocab.keys())[index]



In [183]:
prediction(model, vocab, "The sky is")

'The sky is to'

In [191]:
import time

num_tokens = 10
input_text = "mistaken"

for i in range(num_tokens):
  output_text = prediction(model, vocab, input_text)
  print(output_text)
  input_text = output_text
  # time.sleep(0.5)


mistaken .
mistaken . the
mistaken . the adventure
mistaken . the adventure of
mistaken . the adventure of the
mistaken . the adventure of the speckled
mistaken . the adventure of the speckled band
mistaken . the adventure of the speckled band .
mistaken . the adventure of the speckled band . ''
mistaken . the adventure of the speckled band . '' i


In [47]:
dataloader1 = DataLoader(dataset, batch_size=32, shuffle=False)

In [48]:
# Function to calculate accuracy
def calculate_accuracy(model, dataloader, device):
    model.eval()  # Set the model to evaluation mode
    correct = 0
    total = 0

    with torch.no_grad():  # No need to compute gradients
        for batch_x, batch_y in dataloader1:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)

            # Get model predictions
            outputs = model(batch_x)

            # Get the predicted word indices
            _, predicted = torch.max(outputs, dim=1)

            # Compare with actual labels
            correct += (predicted == batch_y).sum().item()
            total += batch_y.size(0)

    accuracy = correct / total * 100
    return accuracy

# Compute accuracy
accuracy = calculate_accuracy(model, dataloader, device)
print(f"Model Accuracy: {accuracy:.2f}%")


Model Accuracy: 95.65%
